# Local Lipschitz ratio from the perturbed vocab sweep

Cleaned-up reproduction of `playground.ipynb`'s analysis, pointed at a real, currently-available `scripts.generate_perturbed_sweep` output (`onnx_models/simple_stories/perturbed_sweep`) instead of the old single `perturbed_sweep_small.h5` file.

For one sublayer `src -> dst` and one `(sample_idx, pos)` cell, the sweep gives one point per vocab substitution: `data_src[v]`, `data_dst[v]` for `v` in the vocab. This computes pairwise distances *within* that cloud (`pdist`) in both the `src` and `dst` spaces, and the ratio

```
g(x, y) = ||f(x) - f(y)|| / ||x - y||
```

for every pair `(x, y)` -- a local estimate of the (co-)Lipschitz constant of the sublayer map `f = src -> dst`. The minimum of `g` over all pairs is a lower bound on how much `f` expands distances by, at least.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist

from autoencoder.data import load_sample_cloud, sweep_dir_for
from config import load_config
from scripts.estimate_colipschitz_gpd import fit_gpd_tail

## Config

In [ ]:
CONFIG_PATH = "configs/simple_stories.toml"
SRC_LAYER = "add_34"
DST_LAYER = "add_36"
SAMPLE_IDX = 0
POS = 0  # sequence position within the sweep to analyze
N_PAIRS_MAX = 20000  # subsample this many points from the vocab cloud before pdist (pdist is O(n^2))

cfg = load_config(CONFIG_PATH)
sweep_dir = sweep_dir_for(cfg)
print(f"sweep_dir: {sweep_dir}")

## Load the src/dst point clouds for one (sample_idx, pos) cell

`load_sample_cloud` returns `(seq, n_vocab, d_model)` for one sample -- row order is `(sample_idx, pos, vocab_idx)`-ravelled (see `scripts/generate_perturbed_sweep.build_chunk_rows`), so this is a plain reshape+index, no filtering.

In [ ]:
src_cloud = load_sample_cloud(sweep_dir, SRC_LAYER, SAMPLE_IDX)[POS].numpy()  # (n_vocab, d_model)
dst_cloud = load_sample_cloud(sweep_dir, DST_LAYER, SAMPLE_IDX)[POS].numpy()  # (n_vocab, d_model)
print("src_cloud:", src_cloud.shape, " dst_cloud:", dst_cloud.shape)

n_vocab = src_cloud.shape[0]
if n_vocab > N_PAIRS_MAX:
    idx = np.random.default_rng(0).choice(n_vocab, N_PAIRS_MAX, replace=False)
    src_cloud, dst_cloud = src_cloud[idx], dst_cloud[idx]
print("using", src_cloud.shape[0], "points")

## Pairwise distances and the Lipschitz ratio

`pdist` returns the condensed (upper-triangular, no diagonal/duplicates) pairwise-distance vector; `src_cloud`/`dst_cloud` share the same point ordering (both indexed by vocab id), so pair `i` in one lines up with pair `i` in the other -- the ratio is elementwise, no re-indexing needed.

In [ ]:
input_dist = pdist(src_cloud)
output_dist = pdist(dst_cloud)
print(f"{input_dist.size} pairs")

plt.figure(figsize=(6, 6))
plt.scatter(input_dist, output_dist, s=2, alpha=0.2)
plt.xlabel("||x - y|| (input)")
plt.ylabel("||f(x) - f(y)|| (output)")
plt.title(f"{SRC_LAYER} -> {DST_LAYER}, pos={POS}: output vs input distance")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
g = output_dist / input_dist

plt.figure(figsize=(8, 5))
plt.hist(g, bins=100, color="steelblue", edgecolor="black", alpha=0.7)
plt.xlabel("g = ||f(x)-f(y)|| / ||x-y||")
plt.ylabel("count")
plt.title(f"{SRC_LAYER} -> {DST_LAYER}, pos={POS}: local Lipschitz ratio (N={g.size})")
plt.grid(True, alpha=0.3)
plt.show()

print("g min/median/max:", g.min(), np.median(g), g.max())

## GPD tail fit -> estimated co-Lipschitz lower bound

Reuses `scripts.estimate_colipschitz_gpd.fit_gpd_tail` (fixed sign bug: `c_hat = u + scale/shape`, verified numerically against a synthetic ground-truth case -- see the function's docstring/module docstring for the derivation).

In [ ]:
result = fit_gpd_tail(g, threshold=None, threshold_quantile=None, n_tail_min=200)
for k, v in result.items():
    print(f"{k}: {v}")

In [ ]:
threshold = result["threshold"]
exceedances = threshold - g[g < threshold]

fig, (ax_full, ax_tail) = plt.subplots(1, 2, figsize=(13, 5))

ax_full.hist(g, bins=100, color="steelblue", alpha=0.7)
ax_full.axvline(threshold, color="orange", linestyle="--", label=f"threshold u={threshold:.4g}")
if result["c_hat"] is not None:
    ax_full.axvline(result["c_hat"], color="red", linestyle="--", label=f"c_hat={result['c_hat']:.4g}")
ax_full.set_xlabel("g")
ax_full.set_ylabel("count")
ax_full.set_title("full distribution")
ax_full.legend()
ax_full.grid(True, alpha=0.3)

from scipy.stats import genpareto
xi, sigma = result["shape_xi"], result["scale_sigma"]
xs = np.linspace(0, exceedances.max(), 300)
ax_tail.hist(exceedances, bins=30, density=True, color="steelblue", edgecolor="black", alpha=0.6, label="exceedances (empirical)")
ax_tail.plot(xs, genpareto.pdf(xs, xi, loc=0, scale=sigma), color="red", label=f"fitted GPD (xi={xi:.3g})")
ax_tail.set_xlabel("exceedance (u - g)")
ax_tail.set_ylabel("density")
ax_tail.set_title("left tail (below threshold)")
ax_tail.legend()
ax_tail.grid(True, alpha=0.3)

fig.suptitle(f"{SRC_LAYER} -> {DST_LAYER}, pos={POS}")
fig.tight_layout()
plt.show()